In [ ]:
# ─── 1. CÀI ĐẶT MÔI TRƯỜNG (KHÔNG DÙNG THƯ VIỆN ADAPTERS ĐỂ TRÁNH LỖI) ───────
!pip uninstall -y transformers peft accelerate -q
!pip install -q \
    transformers==4.41.2 \
    peft==0.11.1 \
    accelerate==0.30.1 \
    datasets \
    sentencepiece \
    safetensors

# ─── 2. SETUP & IMPORTS ───────────────────────────────────────────────────────
import os, time, functools
import torch
import numpy as np
from PIL import Image
from torch import nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import (CLIPProcessor, CLIPModel, TrainingArguments,
                          Trainer, EarlyStoppingCallback)
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers.trainer_utils import get_last_checkpoint
from datasets import load_dataset
from google.colab import drive

# Tối ưu bộ nhớ GPU
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Kết nối Drive
drive.mount('/content/drive')



# ─── 3. DATASET & PREPROCESS ──────────────────────────────────────────────────
print("⏳ Loading dataset...")
hf_dataset = load_dataset("anson-huang/mirage-news")

model_name = "openai/clip-vit-base-patch32"
processor  = CLIPProcessor.from_pretrained(model_name)

def preprocess(example):
    try:
        image = example['image'].convert("RGB")
    except:
        image = Image.new("RGB", (224, 224), (128, 128, 128))
    inputs = processor(
        text=str(example['text']),
        images=image,
        truncation=True,
        padding="max_length",
        max_length=77,
        return_tensors="pt"
    )
    return {
        "input_ids":      inputs["input_ids"][0],
        "attention_mask": inputs["attention_mask"][0],
        "pixel_values":   inputs["pixel_values"][0],
        "labels":         int(example["label"])
    }

print("⏳ Preprocessing dataset...")
encoded_dataset = hf_dataset.map(
    preprocess,
    remove_columns=hf_dataset["train"].column_names,
    desc="Preprocessing"
)
encoded_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "pixel_values", "labels"]
)

train_dataset = encoded_dataset["train"]
val_dataset   = encoded_dataset["validation"]
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

# ─── 4. MODEL ARCHITECTURE (MANUAL ADAPTER + GATED FUSION + ALIGNMENT) ────────
class Adapter(nn.Module):
    def __init__(self, dim, bottleneck=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, bottleneck),
            nn.GELU(),
            nn.Linear(bottleneck, dim)
        )
    def forward(self, x):
        return x + self.net(x)

class CLIPWithAdapter(nn.Module):
    def __init__(self, model_name, num_labels=2, bottleneck=64, lambda_weight=0.1, delta=0.5):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name)

        # 1. Đóng băng Base Model
        for param in self.clip.parameters():
            param.requires_grad = False

        text_dim = self.clip.config.text_config.hidden_size
        img_dim  = self.clip.config.vision_config.hidden_size
        embed_dim = self.clip.config.projection_dim # 512

        # 2. Manual Adapters
        self.text_adapters = nn.ModuleList([
            Adapter(text_dim, bottleneck)
            for _ in range(self.clip.config.text_config.num_hidden_layers)
        ])
        self.img_adapters = nn.ModuleList([
            Adapter(img_dim, bottleneck)
            for _ in range(self.clip.config.vision_config.num_hidden_layers)
        ])

        # 3. Gated Fusion & Alignment Head (Giữ nguyên cấu trúc so sánh)
        self.cross_attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=8, batch_first=True)
        self.W_g = nn.Linear(embed_dim * 2, embed_dim)
        self.classifier = nn.Linear(embed_dim, num_labels)

        self.lambda_weight = lambda_weight
        self.delta = delta
        self.align_loss_fn = nn.CosineEmbeddingLoss(margin=delta)

    def forward(self, input_ids=None, attention_mask=None, pixel_values=None, labels=None):
        # --- Text Pathway ---
        text_outputs = self.clip.text_model(
            input_ids=input_ids, attention_mask=attention_mask,
            output_hidden_states=True, return_dict=True
        )
        text_hidden = text_outputs.last_hidden_state
        for adapter in self.text_adapters:
            text_hidden = adapter(text_hidden)

        text_embeds = self.clip.text_projection(
            text_hidden[torch.arange(text_hidden.shape[0]), input_ids.argmax(dim=-1)]
        )
        h_T = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

        # --- Image Pathway ---
        vision_outputs = self.clip.vision_model(
            pixel_values=pixel_values, output_hidden_states=True, return_dict=True
        )
        img_hidden = vision_outputs.last_hidden_state
        for adapter in self.img_adapters:
            img_hidden = adapter(img_hidden)

        img_embeds = self.clip.visual_projection(img_hidden[:, 0, :])
        h_I = img_embeds / img_embeds.norm(dim=-1, keepdim=True)

        # --- Gated Modality Fusion ---
        attn_out, _ = self.cross_attention(h_T.unsqueeze(1), h_I.unsqueeze(1), h_I.unsqueeze(1))
        z = attn_out.squeeze(1)

        gate = torch.sigmoid(self.W_g(torch.cat([h_T, h_I], dim=1)))
        h_f = gate * z

        # --- Classification & Loss ---
        logits = self.classifier(h_f)
        loss = None
        if labels is not None:
            loss_cls = nn.CrossEntropyLoss()(logits, labels)
            loss_align = self.align_loss_fn(h_T, h_I, labels.float() * 2 - 1)
            loss = loss_cls + (self.lambda_weight * loss_align)

        return SequenceClassifierOutput(loss=loss, logits=logits)

import gc; gc.collect(); torch.cuda.empty_cache()
model = CLIPWithAdapter(model_name).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Total params    : {total_params:,}")
print(f"📊 Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")

# ─── 5. METRICS & TRAINING ────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds  = np.argmax(probs, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": p,
        "recall":    r,
        "f1":        f1,
        "auc":       roc_auc_score(labels, probs[:, 1])
    }

torch.load = functools.partial(torch.load, weights_only=False)

training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=1e-4, # Giữ mức 1e-4 vì kiến trúc Adapter cần LR này để học tốt hơn
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 21.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
adapters 0.1.2 requires transformers~=4.36.0, but you have transformers 4.41.2 which is incompatible.
Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ Loading dataset...
⏳ Preprocessing dataset...


Preprocessing:   0%|          | 0/10000 [00:00<?, ? examples/s]

Preprocessing:   0%|          | 0/2500 [00:00<?, ? examples/s]

Preprocessing:   0%|          | 0/500 [00:00<?, ? examples/s]

Preprocessing:   0%|          | 0/500 [00:00<?, ? examples/s]

...

Preprocessing:   0%|          | 0/500 [00:00<?, ? examples/s]

Preprocessing:   0%|          | 0/500 [00:00<?, ? examples/s]

Train: 10000 | Val: 2500

📊 Total params    : 154,836,739
📊 Trainable params: 3,559,426 (2.30%)

🚀 Bắt đầu huấn luyện...
▶️ Resuming from: /content/drive/MyDrive/CLIP_ViT_B32_Adapter_MirageNews/checkpoint-3125


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  deprecated_dl_args["even_batches"] = even_batches


RuntimeError: Error(s) in loading state_dict for CLIPWithAdapter:
	size mismatch for classifier.weight: copying a param with shape torch.Size([2, 1024]) from checkpoint, the shape in current model is torch.Size([2, 512]).

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/CLIP_ViT_B32_Adapter_MirageNews_Part2"
os.makedirs(SAVE_DIR, exist_ok=True)
print("\n🚀 Bắt đầu huấn luyện...")
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
start_train = time.time()

last_checkpoint = get_last_checkpoint(SAVE_DIR)
if last_checkpoint:
    print(f"▶️ Resuming from: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("🆕 Training from scratch")
    trainer.train()

trainer.save_model(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
train_time_min = (time.time() - start_train) / 60
print(f"✅ Đã lưu mô hình tại: {SAVE_DIR} | Thời gian train: {train_time_min:.2f} phút")

# ─── 6. LATENCY + VRAM MEASUREMENT ────────────────────────────────────────────
model.eval()
dummy_image = Image.new("RGB", (224, 224), (128, 128, 128))
dummy_text  = "This is a sample news headline for inference measurement."
inputs = processor(
    text=dummy_text, images=dummy_image,
    truncation=True, padding="max_length",
    max_length=77, return_tensors="pt"
)
input_ids      = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)
pixel_values   = inputs["pixel_values"].to(device)

with torch.no_grad():
    for _ in range(10): # Warmup
        model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)

latencies = []
with torch.no_grad():
    for _ in range(100):
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)
        if device == "cuda": torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
vram = torch.cuda.max_memory_allocated() / 1024**3 if device == "cuda" else 0

print(f"\n{'='*50}")
print(f"⚡ Hardware & Efficiency Stats")
print(f"Latency median  : {np.median(latencies):.2f} ms/sample")
print(f"VRAM (peak)     : {vram:.2f} GB")
print(f"{'='*50}")

# ─── 7. TEST EVALUATION (OOD & IND) ───────────────────────────────────────────
print("\n📊 Evaluating on test sets...")
test_splits = ["test1_nyt_mj", "test2_bbc_dalle", "test3_cnn_dalle", "test4_bbc_sdxl", "test5_cnn_sdxl"]

for split in test_splits:
    results = trainer.predict(encoded_dataset[split])
    logits  = results.predictions
    labels  = results.label_ids
    probs   = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds   = np.argmax(probs, axis=1)

    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    print(f"\n[{split.upper()}]")
    print(f" Accuracy : {accuracy_score(labels, preds)*100:.2f}%")
    print(f" F1 Score : {f1*100:.2f}% | AUC: {roc_auc_score(labels, probs[:, 1]):.4f}")


🚀 Bắt đầu huấn luyện...
🆕 Training from scratch


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.296900,0.152422,0.953200,0.932773,0.976800,0.954279,0.992415
2,0.127700,0.107075,0.965200,0.968574,0.961600,0.965074,0.994550
3,0.095700,0.139603,0.962400,0.941221,0.986400,0.963281,0.996341
4,0.064900,0.142839,0.960400,0.937643,0.986400,0.961404,0.996650


✅ Đã lưu mô hình tại: /content/drive/MyDrive/CLIP_ViT_B32_Adapter_MirageNews_Part2 | Thời gian train: 15.61 phút

⚡ Hardware & Efficiency Stats
Latency median  : 53.89 ms/sample
VRAM (peak)     : 0.81 GB

📊 Evaluating on test sets...



[TEST1_NYT_MJ]
 Accuracy : 97.80%
 F1 Score : 97.79% | AUC: 0.9978



[TEST2_BBC_DALLE]
 Accuracy : 56.60%
 F1 Score : 48.21% | AUC: 0.6451



[TEST3_CNN_DALLE]
 Accuracy : 59.00%
 F1 Score : 50.60% | AUC: 0.6711



[TEST4_BBC_SDXL]
 Accuracy : 73.20%
 F1 Score : 73.73% | AUC: 0.8118



[TEST5_CNN_SDXL]
 Accuracy : 79.40%
 F1 Score : 80.08% | AUC: 0.8719


In [ ]:
from google.colab import runtime

# Sau khi đã lưu xong results.json và model checkpoint...
print("✅ Mọi tác vụ đã hoàn thành. Đang đóng session để tiết kiệm GPU...")

# Lệnh này sẽ ngắt kết nối và xóa runtime hiện tại
runtime.unassign()

✅ Mọi tác vụ đã hoàn thành. Đang đóng session để tiết kiệm GPU...
